# Session 12 — Building a Production-ready Evaluation Pipeline

**Halvard Works maintenance assistant.** Four agents answer an engineer's request (planner,
diagnostics, documentation, maintenance), then an engineer's approval and a work order.

> **The question for today:** *Would you ship version B?*

> **The sentence to take away:** *A deployment decision is only as strong as the weakest measurement under it.*

**You edit exactly one file today: `session-12/my_release12.py`.** Everything else is open for
you to read — start with `session-12/pipeline12.py`, which is the whole pipeline in one file.
Nothing in this notebook calls a model: every cell replays runs saved on disk. It is free, and
everyone gets the same numbers.

Halvard Works is fictional — invented for this course. Not affiliated with any vendor.

## Every term, defined before it is used

| term | in this session it means |
|---|---|
| **version** | one exact build of the assistant: same prompts, same model, same code |
| **candidate** | a new version you are thinking of putting live |
| **pipeline** | the same steps, in the same order, every time: saved answers in, a decision out |
| **metric** | one number, worked out the same way for every version |
| **spread** | how much a number could change if you measured again, shown as a range |
| **engineering metric** | a number about how the agents work: right answer, tokens, time |
| **business metric** | a number the plant manager reads: approvals, work orders, requests resolved |
| **regression** | a question that worked on the old version and fails on the new one |
| **regression suite** | the checks every version must pass on every question before it ships |
| **seeded** | broken on purpose, by code we wrote, so it breaks the same way every time |
| **paired comparison** | comparing two versions question by question, then averaging the differences |
| **no detectable difference** | the interval includes zero: 12 questions were not enough to tell, which is not the same as equal |
| **A/B test** | two versions answering the same requests, compared side by side |
| **monitoring** | watching the live system's numbers every day after it ships |
| **alert** | a rule that wakes a person up when a number moves |
| **drift** | the world around the system changing while the system itself stays the same |
| **digital twin** | a software copy of a real factory, kept in step with it, used to test changes safely |

Each term is also explained again in the cell where it first matters.

## Setup

Run this first. It loads every **version** of the assistant — a version is one exact build:
same prompts, same model, same code — and runs each saved run through the pipeline's first
stage. It takes a few seconds. If it prints four lines with no error, you are ready.

In [ ]:
# [SETUP]
import _path  # noqa: F401  -- must be first; puts shared/, plant/ and session-12/ on the path

import pipeline12 as pl
import versions12 as vs
import version_b12
from bench7 import tracing_off

with tracing_off(True):                 # replays are not real runs: never log them
    A = pl.evaluate_all(vs.load('A'))
    B = pl.evaluate_all(vs.load('B'))
    C = pl.evaluate_all(vs.load('C'))
print('version A runs:', len(A))
print('version B runs:', len(B), '' if B else '(not captured yet -- your instructor will say)')
print('version C runs:', len(C), '(SEEDED: made from A, not run again)')
print('pipeline', pl.__version__)

### The change on the table

Version B is a **candidate** — a new version you are thinking of putting live. It changes one
rule in the planner's instructions. Lines starting `-` were removed; lines starting `+` were
added. Below it: what version A did on the question that started all this.

In [ ]:
# [VERSION_B]
print(version_b12.diff())
print()
hw = [r for r in A if r['row_id'] == 'HW-012']
for r in hw:
    print(f"A, HW-012 rep {r['rep']}:  ", ' -> '.join(r['outputs']['agent_calls']))
print('expected:         ', ' -> '.join(hw[0]['row']['expected_agents']))

## Hands-on 1 — Read the scorecard (12 minutes)

A **pipeline** is the same steps, in the same order, every time: saved answers in, a decision
out. A **metric** is one number, worked out the same way for every version. The **spread** is how
much a number could change if you measured again, shown as a range ("95%: 55% to 88%").

**Before you run this cell, write down on paper:** out of version A's 24 runs, how many got
the right fault, part and action (task success)?

Then run it. Next to every line, write who that number really measures: **the agents**, or
**something else**. Read the `measures` column last, and argue with it.

An **engineering metric** is about how the agents work. A **business metric** is what the
plant manager reads. Look hard at *approval rate* and *engineer acceptance*.

In [ ]:
# [SCORECARD]
print(pl.show_scorecard(pl.scorecard(A), 'version A: 24 real runs'))
print()
print('WHERE THE MONEY GOES')
print(pl.cost_by_agent(A))

### Write `resolved`, then screen it

The plant manager wants a *maintenance resolution rate*. Nobody has said what "resolved"
means. Open `session-12/my_release12.py`, Part 1, and write `resolved(run)`. Save, then run
this cell. It shows your number next to three other definitions of the same thing, on the
same 24 runs.

In [ ]:
# [SCREEN_1]
import runpy, sys
sys.argv = ['screen_my_release.py', '--part', '1']
try:
    runpy.run_path(str(_path.ROOT / 'session-12' / 'screen_my_release.py'), run_name='__main__')
except SystemExit:
    pass

## Hands-on 2 — Build the regression suite (20 minutes)

A **regression** is a question that worked on the old version and fails on the new one. A
**regression suite** is the checks every version must pass on every question before it
ships.

The cell below prints, for version A, which agents ran on each question and which were
expected. Find the questions where they differ. Those are the ones your suite must catch.

In [ ]:
# [SUITE]
seen = set()
for r in A:
    if r['row_id'] in seen:
        continue
    seen.add(r['row_id'])
    ran = [a for a in r['outputs']['agent_calls'] if a != 'planner']
    want = [a for a in r['row']['expected_agents'] if a != 'planner']
    mark = '  ' if ran == want else '<>'
    print(f"{mark} {r['row_id']}  ran {ran}")
    if ran != want:
        print(f"            expected {want}   forbidden {r['row']['forbidden_agents']}")
    print(f"            outcome_match = {r['scores']['outcome_match']}")

### Write `must_hold`, then screen it

In `my_release12.py`, Part 2, write `must_hold(run)`: the right agents in the right order,
none of the forbidden ones, and the right answer. Save, then run this cell.

The screener checks three things. Your suite must **fail** on HW-012 (a suite that passes a
known bug cannot catch a new one). It must stay **quiet** on the 8 questions A gets right.
And it must catch **version C**, which is **seeded** — broken on purpose, by code we wrote:
the documentation agent is deleted from every one of A's saved runs.

In [ ]:
# [SCREEN_2]
import runpy, sys
sys.argv = ['screen_my_release.py', '--part', '2']
try:
    runpy.run_path(str(_path.ROOT / 'session-12' / 'screen_my_release.py'), run_name='__main__')
except SystemExit:
    pass

## Hands-on 3 — Ship, hold or reject (20 minutes)

A **paired comparison** lines two versions up question by question, then averages the
differences. **No detectable difference** means the interval includes zero: 12 questions
were not enough to tell — which is not the same as equal.

This cell compares A with C, then A with B, using **your** `must_hold` as the regression
suite. Read C first: what happened to task success? To the suite?

In [ ]:
# [COMPARE]
import importlib, my_release12
importlib.reload(my_release12)
try:
    my_release12.must_hold(A[0]); SUITE = my_release12.must_hold
except NotImplementedError:
    SUITE = None; print('(must_hold not written yet: comparing metrics only)')
print(pl.show_compare(pl.compare(A, C, SUITE)))
print()
if B:
    print(pl.show_compare(pl.compare(A, B, SUITE)))
else:
    print('version B not captured yet')

### Write `decide`, then screen it

In `my_release12.py`, Part 3, write `decide(cmp)`: return `"SHIP"`, `"HOLD"` or `"REJECT"` and
a reason a manager can read. The screener runs it on five situations real release teams have
been in, then on version C. The last line is what your rule says about version B — nobody
knows the right answer in advance. Be ready to defend yours.

In [ ]:
# [SCREEN_3]
import runpy, sys
sys.argv = ['screen_my_release.py', '--part', '3']
try:
    runpy.run_path(str(_path.ROOT / 'session-12' / 'screen_my_release.py'), run_name='__main__')
except SystemExit:
    pass

## Hands-on 4 — A week in production (10 minutes)

**Monitoring** is watching the live system's numbers every day after it ships. An **alert** is a
rule that wakes a person up when a number moves.

Below is a week of production: each day, 24 real requests through version A. The agents do not
change all week. **One thing** does. Find which number moved, and on which day — and notice
which numbers jumped around on days when nothing changed at all.

In [ ]:
# [WEEK]
with tracing_off(True):
    WEEK = [pl.day_metrics(d) for d in vs.production_week()]
print(pl.show_week(WEEK))

### Write `alert`, then screen it

In `my_release12.py`, Part 4, write `alert(today, usual)`. Quiet on normal days; loud when
something changed. An alert that fires on a normal day teaches people to ignore it.

In [ ]:
# [SCREEN_4]
import runpy, sys
sys.argv = ['screen_my_release.py', '--part', '4']
try:
    runpy.run_path(str(_path.ROOT / 'session-12' / 'screen_my_release.py'), run_name='__main__')
except SystemExit:
    pass

## Last: the same pipeline, somewhere else

Your capstone is a **digital twin** — a software copy of a real factory, kept in step with it,
used to test changes safely. Nothing about it is built yet. On paper: for each stage of
today's pipeline, what would it be for a Virtual Fab? What would *its* approval rate secretly
measure?